In [ ]:
import numpy as np
import xarray as xr
import os

In [ ]:
#Extract Data from NetCDF Files

def get_nc_var(file, var_name, start=None, count=None):
    """Extracts a variable from a NetCDF file with optional slicing."""
    with xr.open_dataset(file) as ds:
        if start is not None and count is not None:
            slices = tuple(slice(s, s + c) for s, c in zip(start, count))
            return ds[var_name].isel({dim: s for dim, s in zip(ds[var_name].dims, slices)}).values
        return ds[var_name].values

In [ ]:
def sum_temperature(case_conf):
    
    #Constants
    year_start, year_end = 2007, 2017  # Start and end years
    MS, ME = 1, 12       # Start and end months
    day_start = 1        # Start day
    count_start = 0      # Counter for averaging
    
    #Filepaths
    mesh_hgr = "/mnt/storage0/jmarson/ANALYSES/MASKS/ANHA4_mesh_zgr.nc"
    mesh_zgr = mesh_hgr

    
    # Simulation paths dictionary
    case_dict = {"ANHA4-EJM010-S": ('/mnt/storage0/jmarson/NEMO/ANHA4/ANHA4-EJM010-S', 'VS'),
                 "ANHA4-EJM012-S": ('/mnt/storage0/jmarson/NEMO/ANHA4/ANHA4-EJM012-S', 'VD'),}
    
    
    if case_conf in case_dict:
        data_p, title_str = case_dict[case_conf]
    else:
        raise ValueError(f"Invalid case configuration: {case_conf}")
        
    data_p, title_str = case_dict[case_conf]
    case_conf_clean = case_conf.replace("-S", "")

    print(f"Using data path: {data_p}")
    print(f"Simulation title: {title_str}")

    min_x, min_y = 0, 0
    max_x, max_y = 544, 800
    x_count, y_count = max_x - min_x, max_y - min_y
    
    #Start Temp Sum
    temperature = 0
    count = count_start
    n_files = 0
    
    #Loop Over Time
    for year in range(year_start, year_end + 1):
        for month in range(MS, ME + 1):
            for day in range(day_start, 32):  # Maximum 31 days, adjusted later
                time_tag = f"y{year}m{month:02d}d{day:02d}"
                file_path_v = f"{data_p}/{case_conf_clean}_{time_tag}_gridT.nc"
                if os.path.exists(file_path_v):  # Ensure file exists before using it
                    n_files += 1
                    print(n_files)
                    v = get_nc_var(file_path_v, "votemper", [0, 0, min_y, min_x], [1, 49, y_count, x_count])
                    v = v[0]
                    temp = v
                    temperature += temp
                    
                    np.savez(f"/mnt/storage0/gabriela/ANALYSES/SCRIPTS/data_from_scripts/Temperature/{case_conf}_Average_Temperature_{year_start}{MS}_{year_end}{ME}.npz", Sum_temperature=temperature)


In [ ]:
sum_temperature("ANHA4-EJM010-S")

In [ ]:
sum_temperature("ANHA4-EJM012-S")

In [ ]:
make_conv_e50l("ANHA4-EJM011-S")

In [ ]:
make_conv_e50l("ANHA4-EJM012-S")